# Getting started with Megamicros objects

The `Megamicros` class is the base class for all *Megamicros* usb antennas. 
This Notebook shows how you can use the `Megamicros` interface provided you have a *Megamicros* device connected on your USB port.

Beware that you need to have `libusb` installed, both the python interface and the C libusb.1.0.26 library

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros_tools.log import log
from megamicros.core.mu import Megamicros

log.setLevel( "INFO" )

## Check Usb device and Megamicros device

In [2]:
Megamicros.check_device()
Megamicros.selfest()

2024-06-25 18:10:01,756 [INFO]:  .Checking usb devices...
2024-06-25 18:10:01,773 [INFO]:  .Found Megamicros [Mu32] device
2024-06-25 18:10:01,774 [INFO]:  .Connected on USB device fe27:ac03
2024-06-25 18:10:01,775 [INFO]:  .Found following device fe27:ac03 characteristics :
2024-06-25 18:10:01,775 [INFO]:   > OS System: Darwin
2024-06-25 18:10:01,775 [INFO]:   > Bus number: 1
2024-06-25 18:10:01,776 [INFO]:   > Ports number: 1
2024-06-25 18:10:01,776 [INFO]:   > Device address: 5 (0005)
2024-06-25 18:10:01,778 [INFO]:   > Device name: M? 
2024-06-25 18:10:01,779 [INFO]:   > Manufacturer: DALEMBE
2024-06-25 18:10:01,780 [INFO]:   > Serial number: None
2024-06-25 18:10:01,780 [INFO]:   > Device speed:  [SUPER SPEED] (The device is operating at high speed (480MBit/s))
2024-06-25 18:10:01,783 [INFO]:  .Checking megamicros device...
2024-06-25 18:10:01,786 [INFO]:  .Installing Megamicros settings...
2024-06-25 18:10:01,791 [INFO]:  .Created a new antenna
2024-06-25 18:10:01,812 [INFO]:  .C

([0, 1, 2, 3, 4, 5, 6, 7], [0, 1])

## Declare your antenna object

The ``Megamicros`` constructor needs no argument. The device is detected and the class device parameters are populated.

In [4]:
# Define an empty antenna
antenna = Megamicros()

2024-06-25 18:10:24,560 [INFO]:  .Installing Megamicros settings...
2024-06-25 18:10:24,561 [INFO]:  .Created a new antenna
2024-06-25 18:10:24,562 [INFO]:  .Checking usb devices...
2024-06-25 18:10:24,574 [INFO]:  .Found Megamicros [Mu32] device
2024-06-25 18:10:24,575 [INFO]:  .Connected on USB device fe27:ac03
2024-06-25 18:10:24,575 [INFO]:  .Found following device fe27:ac03 characteristics :
2024-06-25 18:10:24,576 [INFO]:   > OS System: Darwin
2024-06-25 18:10:24,576 [INFO]:   > Bus number: 1
2024-06-25 18:10:24,576 [INFO]:   > Ports number: 1
2024-06-25 18:10:24,577 [INFO]:   > Device address: 5 (0005)
2024-06-25 18:10:24,577 [INFO]:   > Device name: M? 
2024-06-25 18:10:24,578 [INFO]:   > Manufacturer: DALEMBE
2024-06-25 18:10:24,578 [INFO]:   > Serial number: None
2024-06-25 18:10:24,578 [INFO]:   > Device speed:  [SUPER SPEED] (The device is operating at high speed (480MBit/s))
2024-06-25 18:10:24,579 [INFO]:  .Performing Megamicros selftest on 0.1 s (5000 samples) ...
2024-0

Lets know some properties of the connected antenna:

In [5]:
print( f"Available MEMs = {antenna.available_mems}" )
print( f"Available analogs = {antenna.available_analogs}" )
print( f"System type = {antenna.system_type}" )

Available MEMs = [0, 1, 2, 3, 4, 5, 6, 7]
Available analogs = [0, 1]
System type = Mu32


Try 1s of acquisition on all available MEMs

In [ ]:
antenna.run( 
    duration=1, 
    mems = [0, 1],
    analogs = [0, 1],
    counter=False, 
    counter_skip=False, 
    status=False,
    datatype = 'int32',
    sampling_frequency=50000
)

# Wait until the thread terminates
antenna.wait()

One can force some MEMs to be available while they actually are not. 
Corresponding channels will stay at 0.

In [ ]:
antenna.setAvailableMems( [i for i in range(32)] )

antenna.run( 
    duration=1, 
    mems = [0, 10],
    counter=False, 
    counter_skip=False, 
    status=False,
    datatype = 'int32',
    sampling_frequency=50000
)

antenna.wait()


Getting data can be done by iterate through the `antenna` object provided the antenna had already recived all the data:

In [ ]:
# get data
for data in antenna:
    print( f"data={data}" )

The previous method suggests that real time is not possible.
Actually, you can perform real time acqusition if you iterate before calling the `wait` method, just after the `run` call:   

In [ ]:
import time

antenna.run( 
    duration=1, 
    mems = [0, 1],
    counter=False, 
    counter_skip=False, 
    status=False,
    sampling_frequency=50000
)

# get data
start_time = time.time()
for data in antenna:
    print( f"data={data}" )

print( f"Elapsed time= {time.time() - start_time}" )

# Wait until the thread terminates
antenna.wait()

Note that the elapsed time reaches 4 seconds, whereas the requested acquisition time was 1 second.
Moreover, the system message reported 0.94s.

This is because the system's time measurement method is more accurate: it only includes the actual acquisition time.

The user method, as reported above, includes, in addition to the acquisition:
* The MEMs initialization delay (MEMs powering), default is 1 second
* The timeout delay to leave the queue (2 seconds)
As a consequence, the total elapsed is 4 secondes.

## Plots

See the `mu_graph.py` example in the èxamples` directory

## Saving data in H5 file

In [ ]:
# Define an empty antenna
antenna = Megamicros()

# Set all MEMs as available
antenna.setAvailableMems( [i for i in range(32)] )

antenna.run( 
    duration=10, 
    mems = antenna.available_mems,
    counter=False, 
    counter_skip=False, 
    status=False,
    sampling_frequency=50000,
    h5_recording=True,                          # H5 recording ON
    h5_rootdir='./',                            # directory where to save file
    h5_filename='test.h5',                      # filename
)

antenna.wait()

### Control the H5 file created

We can control easily the H5 file generated by building a new antenna with this H5 file as input. 

In [ ]:
from megamicros.core.h5 import MemsArrayH5

# Define the antenna
antenna_h5 = MemsArrayH5( 
    filename='test.h5',
)

print( f"Sampling frequency: {antenna_h5.sampling_frequency}Hz" )
print( f"Channels number: {antenna_h5.mems_number}" )
print( f"Available MEMs number: {antenna_h5.available_mems_number}" )
print( f"Available MEMs: {antenna_h5.available_mems}" )
print( f"Whether counter is available or not: {antenna_h5.counter}" )
print( f"Dataset(s) number: " + str(antenna_h5.dataset_number) )


### Plot signals

In [ ]:
# Reinit the antenna
antenna_h5 = MemsArrayH5( 
    filename='test.h5',
)

# Choose MEMS to plot
MEMS = [0, 1]

# Init a np.ndarray
signals = np.ndarray( (0, len(MEMS) ) )

# Run the H5 antenna
antenna_h5.run( 
    duration=1, 
    mems = MEMS,
    real_time=False,
)

# Get signals
for data in antenna_h5:
    print( f"data shape={np.shape(data)}")
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
antenna_h5.wait()
print( f"exit from loop. Signal shape is: {np.shape( signals )}" )

In [ ]:
# plot signals
time = np.array( range( np.size(signals,0) ) )/antenna_h5.sampling_frequency
#time = np.array( range( 1000 ) )/antenna_h5.sampling_frequency
fig, axs = plt.subplots( antenna_h5.channels_number )
fig.suptitle('Channels activity')	
for s in range( antenna_h5.channels_number ):
    axs[s].plot( time, signals[:len(time),s] )
    axs[s].set( xlabel='time in seconds', ylabel='channel %d' % s )

plt.show()